# Track 2 Tier 1 G1 retargeting probe
Retarget the first 32 frames of released episode 0 using the official SOMA-X/Pink
pipeline. This validates the challenge data adapter; it is not a full submission.
No SAM 3D Body checkpoint or token is required for this Tier 1 path.

In [ ]:
import os,sys,subprocess,json,hashlib,shutil,urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run(['uv','python','install','3.11'],check=True)
subprocess.run(['uv','venv','--python','3.11','/tmp/retarget-env'],check=True)
py='/tmp/retarget-env/bin/python'
deps=['torch','py-soma-x==0.2.1','pin==3.7.0','pin-pink==4.2.0','coal==3.0.1',
'eigenpy==3.10.3','cmeel==0.57.3','cmeel-boost==1.87.0.1','cmeel-urdfdom==4.0.1',
'cmeel-console-bridge==1.0.2.3','cmeel-tinyxml2==10.0.0','cmeel-tinyxml==2.6.2.3',
'cmeel-assimp==5.4.3.1','cmeel-octomap==1.10.0','cmeel-qhull==8.0.2.1',
'cmeel-zlib==1.3.1','qpsolvers[daqp]==4.9.0','loop-rate-limiters','viser',
'trimesh<5','scipy','pandas','pyarrow','huggingface_hub','usd-core']
subprocess.run(['uv','pip','install','--python',py,*deps],check=True)
repo=Path('/tmp/video_to_data');rev='8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef'
env=os.environ.copy();env['GIT_LFS_SKIP_SMUDGE']='1'
subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/nvidia-isaac/video_to_data.git',str(repo)],check=True,env=env)
subprocess.run(['git','sparse-checkout','set','robotic_grounding'],cwd=repo,check=True,env=env)
subprocess.run(['git','checkout',rev],cwd=repo,check=True,env=env)
rg=repo/'robotic_grounding';assets=rg/'source/robotic_grounding/robotic_grounding/assets'
env['PYTHONPATH']=str(rg/'source/robotic_grounding')


In [ ]:
pointers=[]
for f in (assets/'meshes/g1').rglob('*'):
 if f.is_file() and f.stat().st_size<1024:
  raw=f.read_bytes()
  if raw.startswith(b'version https://git-lfs.github.com/spec/v1'):
   fields=dict(line.split(' ',1) for line in raw.decode().splitlines())
   pointers.append((f,fields['oid'].split(':')[1],int(fields['size'])))
def fetch(item):
 f,digest,size=item
 url='https://media.githubusercontent.com/media/nvidia-isaac/video_to_data/'+rev+'/'+f.relative_to(repo).as_posix()
 tmp=f.with_suffix(f.suffix+'.download');h=hashlib.sha256()
 with urllib.request.urlopen(url,timeout=180) as response,tmp.open('wb') as out:
  while chunk:=response.read(1024*1024):h.update(chunk);out.write(chunk)
 assert tmp.stat().st_size==size and h.hexdigest()==digest
 tmp.replace(f)
with ThreadPoolExecutor(max_workers=4) as pool:list(pool.map(fetch,pointers))
subprocess.run([py,'scripts/setup_soma_assets.py'],cwd=rg,env=env,check=True)


In [ ]:
source=Path('/tmp/tier1');(source/'meta').mkdir(parents=True)
base='https://huggingface.co/datasets/nvidia/video_to_data_challenge/resolve/main/track_2/tier_1_multiview_caption/'
def get(relative):
 dest=source/relative;dest.parent.mkdir(parents=True,exist_ok=True)
 urllib.request.urlretrieve(base+relative,dest)
 return dest
get('meta/info.json');metadata=get('meta/episodes_metadata.jsonl')
items=[json.loads(line) for line in metadata.read_text().splitlines()]
m=next(x for x in items if x['episode_index']==0)
metadata.write_text(json.dumps(m)+'\n')
for relative in ['data/chunk-000/episode_000000.parquet',m['mesh'],m['ground_plane'],m['support_surface']]:get(relative)


In [ ]:
Path('/tmp/export_inputs.py').write_text('"""Convert one released Track 2 tier to the official SOMA-to-G1 input layout.\n\nNo clean-reference lookup is performed for another tier. Object visibility gaps\nare filled by interpolation solely from that tier\'s observed poses. Human poses\nremain in their released SOMA convention; object poses remain OpenCV world poses,\nas required by soma_to_g1.py\'s explicit CV-to-SOMA transform.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport shutil\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial.transform import Rotation, Slerp\n\np = argparse.ArgumentParser()\np.add_argument(\'--input\', type=Path, required=True)\np.add_argument(\'--output\', type=Path, required=True)\na = p.parse_args()\ninfo = json.loads((a.input/\'meta/info.json\').read_text())\nnames = info[\'features\'][\'observation.human.pose\'][\'names\']\njoint_names = [name.rsplit(\'.\', 1)[0] for name in names[::3]]\nassert len(joint_names) == 77\nfor line in (a.input/\'meta/episodes_metadata.jsonl\').read_text().splitlines():\n    meta = json.loads(line)\n    episode = int(meta[\'episode_index\'])\n    frame = pd.read_parquet(a.input/f\'data/chunk-000/episode_{episode:06d}.parquet\')\n    dest = a.output/meta[\'sequence_id\']\n    # Require the original mesh/ground plane; never silently substitute primitives.\n    mesh = a.input/meta[\'mesh\']\n    plane = a.input/meta[\'ground_plane\']\n    if not mesh.is_file() or not plane.is_file():\n        raise FileNotFoundError(f\'Missing released mesh or ground plane for episode {episode}\')\n    dest.mkdir(parents=True, exist_ok=True)\n    stack = lambda key: np.stack(frame[key]).astype(np.float32)\n    np.savez_compressed(dest/\'soma_params.npz\',\n        poses=stack(\'observation.human.pose\').reshape(-1,77,3),\n        transl=stack(\'observation.human.translation\'),\n        identity_coeffs=stack(\'observation.human.identity_coeffs\'),\n        scale_params=stack(\'observation.human.scale_params\'),\n        bone_length_flexibles=stack(\'observation.human.bone_length_flexibles\'),\n        joint_names=np.asarray(joint_names), identity_model_type=\'mhr\',\n        unit=\'meters\',keep_root=False,absolute_pose=False)\n    pose = stack(\'observation.object.pose\')\n    valid = frame[\'observation.object.visible\'].to_numpy(bool, copy=True)\n    valid &= np.isfinite(pose).all(axis=1) & (np.linalg.norm(pose[:,3:],axis=1)>1e-8)\n    ids = np.flatnonzero(valid)\n    if len(ids)<2:\n        raise ValueError(f\'Insufficient object observations for episode {episode}\')\n    t = frame[\'timestamp\'].to_numpy(float)\n    translation=np.column_stack([np.interp(t,t[ids],pose[ids,i]) for i in range(3)])\n    rotation=Slerp(t[ids],Rotation.from_quat(pose[ids,3:],scalar_first=True))(np.clip(t,t[ids[0]],t[ids[-1]]))\n    transforms=np.tile(np.eye(4,dtype=np.float32),(len(t),1,1))\n    transforms[:,:3,:3]=rotation.as_matrix();transforms[:,:3,3]=translation\n    np.save(dest/\'poses.npy\',transforms)\n    (dest/\'reconstructed_mesh\').mkdir(exist_ok=True)\n    shutil.copy2(mesh,dest/\'reconstructed_mesh/output_aligned.glb\')\n    shutil.copy2(plane,dest/\'ground_plane.json\')\n    (dest/\'source.json\').write_text(json.dumps({\'episode_index\':episode,\n        \'source_tier\':str(a.input.resolve()),\'sequence_id\':meta[\'sequence_id\'],\n        \'fps\':info[\'fps\'],\'interpolated_object_frames\':int((~valid).sum()),\n        \'support_surface\':meta[\'support_surface\']},indent=2)+\'\\n\')\n    print(f\'episode {episode}: {len(frame)} frames; {int((~valid).sum())} missing poses interpolated\')\n')
subprocess.run([py,'/tmp/export_inputs.py','--input',str(source),'--output','/tmp/retarget-inputs'],check=True)

In [ ]:
motion_root=Path('/kaggle/working/motion_assets/human_motion_data')
subprocess.run([py,'scripts/retarget/soma_to_g1.py','/tmp/retarget-inputs/'+m['sequence_id'],
 '--save','--motion-root',str(motion_root),'--soma-subdir','soma','--end-frame','32'],
 cwd=rg,env=env,check=True)
stage=motion_root/'whole_body/reconstructed_stage';stage.mkdir(parents=True,exist_ok=True)
shutil.copy2(source/m['support_surface'],stage/Path(m['support_surface']).name)
parquets=list(motion_root.rglob('*.parquet'));assert parquets
print('Retargeted 32-frame probe:',[str(p) for p in parquets])
